# Section 1 — Load Reuters Dataset and Tokenize Documents

In [2]:
import nltk
from nltk.corpus import reuters
from nltk import word_tokenize

nltk.download('reuters')
nltk.download('punkt_tab')

doc_ids = reuters.fileids()
print("Number of documents:", len(doc_ids))

documents = [(doc_id, reuters.raw(doc_id)) for doc_id in doc_ids]

documents_tokenized = [(doc_id, word_tokenize(text)) for doc_id, text in documents]

print("Sample tokenized document:")
print(documents_tokenized[0])


[nltk_data] Downloading package reuters to /root/nltk_data...
[nltk_data]   Package reuters is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Number of documents: 10788
Sample tokenized document:
('test/14826', ['ASIAN', 'EXPORTERS', 'FEAR', 'DAMAGE', 'FROM', 'U.S.-JAPAN', 'RIFT', 'Mounting', 'trade', 'friction', 'between', 'the', 'U.S.', 'And', 'Japan', 'has', 'raised', 'fears', 'among', 'many', 'of', 'Asia', "'s", 'exporting', 'nations', 'that', 'the', 'row', 'could', 'inflict', 'far-reaching', 'economic', 'damage', ',', 'businessmen', 'and', 'officials', 'said', '.', 'They', 'told', 'Reuter', 'correspondents', 'in', 'Asian', 'capitals', 'a', 'U.S.', 'Move', 'against', 'Japan', 'might', 'boost', 'protectionist', 'sentiment', 'in', 'the', 'U.S.', 'And', 'lead', 'to', 'curbs', 'on', 'American', 'imports', 'of', 'their', 'products', '.', 'But', 'some', 'exporters', 'said', 'that', 'while', 'the', 'conflict', 'would', 'hurt', 'them', 'in', 'the', 'long-run', ',', 'in', 'the', 'short-term', 'Tokyo', "'s", 'loss', 'might', 'be', 'their', 'gain', '.', 'The', 'U.S.', 'Has', 'said', 'it', 'will', 'impose', '300', 'mln', 'dlrs', 'of

# Section 2 — Preprocessing Functions

In [3]:
import re
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('wordnet')

# for the first step of the preprocessing
def apply_lowercase(tokens):
    return [t.lower() for t in tokens]


# for the second step of the preprocessing
stop_50 = set([
    'the', 'is', 'in', 'and', 'to', 'of', 'a', 'for', 'on', 'with',
    'as', 'by', 'at', 'from', 'this', 'that', 'an', 'be', 'it', 'or',
    'are', 'was', 'were', 'but', 'not', 'have', 'has', 'had', 'their', 'they',
    'which', 'you', 'we', 'can', 'will', 'would', 'about', 'more', 'his',
    'her', 'its', 'one', 'all', 'also', 'there', 'up', 'out', 'may', 'so'
])
stop_20 = set([
    'the', 'is', 'in', 'and', 'to', 'of', 'a', 'for', 'on', 'with',
    'as', 'by', 'at', 'from', 'this', 'that', 'an', 'be', 'it', 'or'
])

def remove_stopwords(tokens, stop_set):
    return [t for t in tokens if t not in stop_set]


# for the third step of the preprocessing
ps = PorterStemmer()

def apply_stemming(tokens):
    return [ps.stem(t) for t in tokens]


# for the forth step of the preprocessing
lemmatizer = WordNetLemmatizer()

def apply_lemmatization(tokens):
    return [lemmatizer.lemmatize(t) for t in tokens]


# for the fifth step of the preprocessing
punct_regex = re.compile(r'[^a-zA-Z0-9]')
punct_num_regex = re.compile(r'[^a-zA-Z]')

def remove_punctuation(tokens):
    cleaned_tokens = []
    for t in tokens:
        new_t = punct_regex.sub('', t)
        if new_t != '':
            cleaned_tokens.append(new_t)
    return cleaned_tokens

def remove_punct_and_numbers(tokens):
    cleaned_tokens = []
    for t in tokens:
        new_t = punct_num_regex.sub('', t)
        if new_t != '':
            cleaned_tokens.append(new_t)
    return cleaned_tokens


# for the sixth step of the preprocessing
def compute_doc_frequency(inverted_index):
    return {term: len(postings) for term, postings in inverted_index.items()}

# Lossy pruning (doc freq < 2)
def remove_rare_terms(tokens, rare_terms_set):
    return [t for t in tokens if t not in rare_terms_set]

# Keep top-k terms
def keep_top_k_terms(tokens, top_k_set):
    return [t for t in tokens if t in top_k_set]


[nltk_data] Downloading package wordnet to /root/nltk_data...


# Section 3 — Build Inverted Indexes for All Preprocessing Cases

In [4]:
from collections import defaultdict, Counter

def build_inverted_index(documents_tokens):
    inverted = defaultdict(list)
    for doc_id, tokens in documents_tokens:
        for term in set(tokens):
            inverted[term].append(doc_id)
    return inverted

# for top-k terms
all_tokens = [t for doc_id, tokens in documents_tokenized for t in tokens]
token_freq = Counter(all_tokens)
K = 20000
top_k_terms = set([term for term, freq in token_freq.most_common(K)])

# for Lossy pruning
rare_threshold = 2

indexes = {}

# 1. Baseline
indexes['baseline'] = build_inverted_index(documents_tokenized)

# 2. Lowercasing
docs_lower = [(doc_id, apply_lowercase(tokens)) for doc_id, tokens in documents_tokenized]
indexes['lowercase'] = build_inverted_index(docs_lower)

# 3. Stopword removal (20 words)
docs_stop20 = [(doc_id, remove_stopwords(tokens, stop_20)) for doc_id, tokens in docs_lower]
indexes['stop20'] = build_inverted_index(docs_stop20)

# 4. Stopword removal (50 words)
docs_stop50 = [(doc_id, remove_stopwords(tokens, stop_50)) for doc_id, tokens in docs_lower]
indexes['stop50'] = build_inverted_index(docs_stop50)

# 5. Stemming
docs_stemmed = [(doc_id, apply_stemming(tokens)) for doc_id, tokens in docs_lower]
indexes['stemmed'] = build_inverted_index(docs_stemmed)

# 6. Lemmatization
docs_lemmatized = [(doc_id, apply_lemmatization(tokens)) for doc_id, tokens in docs_lower]
indexes['lemmatized'] = build_inverted_index(docs_lemmatized)

# 7. Remove only punctuation
docs_no_punct = [(doc_id, remove_punctuation(tokens)) for doc_id, tokens in docs_lower]
indexes['no_punct'] = build_inverted_index(docs_no_punct)

# 8. Remove punctuation + numbers
docs_no_punct_num = [(doc_id, remove_punct_and_numbers(tokens)) for doc_id, tokens in docs_lower]
indexes['no_punct_num'] = build_inverted_index(docs_no_punct_num)

# 9. Lossy: remove rare terms ( doc freq < rare_threshold(2) )
rare_terms = set([term for term, df in compute_doc_frequency(indexes['baseline']).items() if df < rare_threshold])
docs_lossy_rare = [(doc_id, remove_rare_terms(tokens, rare_terms)) for doc_id, tokens in docs_lower]
indexes['lossy_rare'] = build_inverted_index(docs_lossy_rare)

# 10. Lossy: top-K frequent terms
docs_lossy_topk = [(doc_id, keep_top_k_terms(tokens, top_k_terms)) for doc_id, tokens in docs_lower]
indexes['lossy_topk'] = build_inverted_index(docs_lossy_topk)

# Section 4 — Compare Index Sizes, Vocab Size, and Posting Bytes

In [8]:
import sys

def compute_index_metrics(inverted_index):
    vocab_size = len(inverted_index)

    posting_bytes = sum(len(postings) * 4 for postings in inverted_index.values())

    index_size_bytes = sys.getsizeof(inverted_index) + posting_bytes

    return index_size_bytes, vocab_size, posting_bytes

metrics = {}
for name, idx in indexes.items():
    metrics[name] = compute_index_metrics(idx)

baseline = metrics['baseline']

options = [
    ('Lowercase only', 'lowercase'),
    ('Stopword removal (20 words)', 'stop20'),
    ('Stopword removal (50 words)', 'stop50'),
    ('Stemming (Porter)', 'stemmed'),
    ('Lemmatization', 'lemmatized'),
    ('puncs removal', 'no_punct'),
    ('puncs & nums removal', 'no_punct_num'),
    ('Lossy pruning', 'lossy_rare'),
    ('Top-K terms', 'lossy_topk')
]

print(f"{'Preprocessing':<30}{'Index Size (bytes)':<25}{'Vocab Size':<20}{'Posting Bytes':<20}")
print(f"{'':<30}{'  Raw    |   ∆(%)':<25}{'Raw  |   ∆(%)':<20}{'Raw   |   ∆(%)':<20}")
print("-"*95)

for display_name, key in options:
    idx_size, vocab, postings = metrics[key]

    idx_delta = 100 * (baseline[0] - idx_size) / baseline[0]
    vocab_delta = 100 * (baseline[1] - vocab) / baseline[1]
    postings_delta = 100 * (baseline[2] - postings) / baseline[2]

    print(f"{display_name:<30}{idx_size:<8} | {idx_delta:6.2f}%   {vocab:<8} | {vocab_delta:6.2f}%   {postings:<8} | {postings_delta:6.2f}%")


Preprocessing                 Index Size (bytes)       Vocab Size          Posting Bytes       
                                Raw    |   ∆(%)        Raw  |   ∆(%)       Raw   |   ∆(%)      
-----------------------------------------------------------------------------------------------
Lowercase only                5282892  |   4.10%   52230    |  16.91%   3360404  |   6.30%
Stopword removal (20 words)   4931640  |  10.48%   52210    |  16.94%   3009152  |  16.10%
Stopword removal (50 words)   4749768  |  13.78%   52181    |  16.98%   2827280  |  21.17%
Stemming (Porter)             5157060  |   6.39%   44685    |  28.91%   3234572  |   9.81%
Lemmatization                 5211068  |   5.41%   49820    |  20.74%   3288580  |   8.31%
puncs removal                 5099432  |   7.43%   48352    |  23.07%   3176944  |  11.42%
puncs & nums removal          3837132  |  30.35%   32073    |  48.97%   2875844  |  19.81%
Lossy pruning                 4206568  |  23.64%   32352    |  48.53%   324

# Section 5 _ Compare the Result of the Queries

In [9]:
queries = [
    "oil AND market",
    "trade AND deficit",
    "gold AND prices",
    "economy AND growth",
    "dollar AND interest",
    "Iran AND Iraq"
]

options = [
    ('Baseline (raw)', 'baseline'),
    ('Lowercase only', 'lowercase'),
    ('Stopword removal (20 words)', 'stop20'),
    ('Stopword removal (50 words)', 'stop50'),
    ('Stemming (Porter)', 'stemmed'),
    ('Lemmatization', 'lemmatized'),
    ('puncs removal', 'no_punct'),
    ('puncs & nums removal', 'no_punct_num'),
    ('Lossy pruning', 'lossy_rare'),
    ('Top-K terms', 'lossy_topk')
]

# Boolean search function from project 1
def boolean_search(query, inverted_index):
    terms = [term.strip() for term in query.split(" AND ")]
    if not all(term in inverted_index for term in terms):
        return []
    doc_lists = [set(inverted_index[term]) for term in terms]
    result = set.intersection(*doc_lists)
    return sorted(result)

results = {}
for display_name, key in options:
    index = indexes[key]
    results[display_name] = []
    print(f"=== {display_name} ===")
    for q in queries:
        retrieved_docs = boolean_search(q, index)
        results[display_name].append(len(retrieved_docs))
        sample_docs = retrieved_docs[:10]  # first 10
        print(f"Query: {q}")
        print(f"Documents retrieved: {len(retrieved_docs)}")
        print(f"Sample docs: {sample_docs}\n")

=== Baseline (raw) ===
Query: oil AND market
Documents retrieved: 178
Sample docs: ['test/14833', 'test/14891', 'test/14892', 'test/15063', 'test/15212', 'test/15322', 'test/15639', 'test/15875', 'test/16093', 'test/16176']

Query: trade AND deficit
Documents retrieved: 224
Sample docs: ['test/14832', 'test/14862', 'test/14931', 'test/15154', 'test/15246', 'test/15310', 'test/15352', 'test/15386', 'test/15442', 'test/15460']

Query: gold AND prices
Documents retrieved: 32
Sample docs: ['test/14852', 'test/14890', 'test/16009', 'test/16072', 'test/16589', 'test/16604', 'test/17714', 'test/17966', 'test/20021', 'training/10546']

Query: economy AND growth
Documents retrieved: 155
Sample docs: ['test/14862', 'test/14891', 'test/14931', 'test/14987', 'test/15063', 'test/15212', 'test/15246', 'test/15372', 'test/15450', 'test/15539']

Query: dollar AND interest
Documents retrieved: 146
Sample docs: ['test/14890', 'test/14931', 'test/15212', 'test/15310', 'test/15364', 'test/15384', 'test/15